# Day 2-03｜動画全体の処理をつなぐ：検出、コートのキーポイント、BEV の統合

[Colab で開く](https://colab.research.google.com/github/henry753951/basketball-hackathon-course/blob/main/day2/d2_03_bbox_to_bev_integration_ja.ipynb)

[English](d2_03_bbox_to_bev_integration_en.ipynb) | [Japanese](d2_03_bbox_to_bev_integration_ja.ipynb) | [Traditional Chinese](d2_03_bbox_to_bev_integration.ipynb)


> 前の単元では 1 枚の画像と固定のホモグラフィを使った。ここでは動画に拡張し、各フレームで選手とコート構造を検出する。  
> 複数のモデルと幾何処理を、一つのパイプラインとして初めて接続する。

## 学習目標

- 各フレームで選手とコートのキーポイントを検出する。
- 現在のキーポイントからホモグラフィを推定し、選手の足元を投影する。
- 元映像と BEV を並べた動画、およびフレームごとの JSON を出力する。
- 現状ではフレーム間の ID がつながっていないことを理解し、Day 3 の追跡につなげる。

## 用語

- **パイプライン（Pipeline）**：前の処理の出力を、次の処理の入力にする一連の計算。
- **フレームごとの処理（Per-frame）**：各画像を独立に処理すること。この段階では同じ人物に共通の ID はない。
- **代替処理（Fallback）**：キーポイントが不足したとき、直前の信頼できる結果を一時的に再利用し、ちらつきを抑えること。


## 実行環境の確認

- Colab では **GPU** ランタイムを選ぶ。本教材の Ultralytics／PyTorch の設定では TPU を直接使用しない。
- CPU でも実行できるが、推論や動画の出力に時間がかかる。最初は `MAX_FRAMES` を小さくして処理を確認する。
- 実行前に各セルの目的を読み、出力が想定どおりか確認する。エラーが出ないことだけで正しいと判断しない。


## 演習の流れ
1. 別の参照動画を選ぶ。
2. 各フレームで物体検出とコートのキーポイント推論を実行する。
3. キーポイントとホモグラフィを組み合わせ、選手の足元を連続的に BEV に投影し、`d2_03_detector_keypoint_bev.mp4` と JSON を作る。


In [1]:
import os
os.environ["BASKETBALL_COURSE_LANGUAGE"] = "ja"

# 教材フォルダを探し、必要なら取得して共通の環境設定を読み込みます。
# まず初期設定で実行し、出力を確認してから一度に一つのパラメータを変えます。
from pathlib import Path
import subprocess
import sys

DRIVE_MOUNTED = False
if "google.colab" in sys.modules:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("この Colab ランタイムでは Drive をマウントできないため、ローカルの /content を使います。")

COURSE_ROOT_HINT = (
    Path("/content/drive/MyDrive/basketball_hackathon/course")
    if DRIVE_MOUNTED
    else next(
        (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
         if (p / "src" / "course_setup.py").exists()),
        Path("/content/basketball_hackathon/course"),
    )
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT, mount_drive=False)


教材ルート： /Users/henry753951/Developer/sandbox/basketball-hackathon-course
素材フォルダ： /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets
共通モジュール： /Users/henry753951/Developer/sandbox/basketball-hackathon-course/src


In [2]:
# 演習の次の処理を実行します。
# まず初期設定で実行し、出力を確認してから一度に一つのパラメータを変えます。
from src.video_utils import display_video_in_notebook
from src.yolo_utils import preferred_court_keypoint_model_path, write_detector_keypoint_bev_video

videos = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if len(videos) < 2:
    raise FileNotFoundError("assets/raw/reference_videos/ に参照動画が 2 本以上必要です。")

VIDEO_PATH = videos[1]
DETECTOR_PATH = COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
COURT_MODEL_PATH = preferred_court_keypoint_model_path(COURSE_ROOT)
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"

print("video:", VIDEO_PATH)
print("detector:", DETECTOR_PATH)
print("court model:", COURT_MODEL_PATH)
print("start frame:", 90)


video: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/raw/reference_videos/boston-celtics-new-york-knicks-game-1-q1-03.16-03.11.mp4
detector: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/yolo26n_basketball_player_best.pt
court model: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/court_keypoints/yolo26n_basketball_court_pose_best.pt
start frame: 90


In [3]:
# 演習の次の処理を実行します。
# まず初期設定で実行し、出力を確認してから一度に一つのパラメータを変えます。
bev_video = COURSE_ROOT / "assets" / "results" / "d2_03_detector_keypoint_bev.mp4"
bev_video, rows = write_detector_keypoint_bev_video(
    video_path=VIDEO_PATH,
    detector_path=DETECTOR_PATH,
    court_model_path=COURT_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=bev_video,
    max_frames=90,
    detector_conf=0.25,
    keypoint_conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=90,
)
bev_json = bev_video.with_suffix(".json")

print("BEV video:", bev_video)
print("BEV json:", bev_json)
print("projected rows:", len(rows))
display_video_in_notebook(bev_video, loop=True)


$ /opt/homebrew/bin/ffmpeg -y -i /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d2_03_detector_keypoint_bev.mp4 -an -vcodec libx264 -pix_fmt yuv420p -movflags +faststart -preset veryfast -crf 23 /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d2_03_detector_keypoint_bev.notebook.mp4


BEV video: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d2_03_detector_keypoint_bev.mp4
BEV json: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d2_03_detector_keypoint_bev.json
projected rows: 605


この BEV の点には、まだフレーム間で共通の ID がない。Day 3 では ByteTrack を加え、同じ選手の track ID を継続させる。

## 出力ファイル

- `assets/results/d2_03_detector_keypoint_bev.mp4`：検出、コートのキーポイント、BEV を並べた動画。
- `assets/results/d2_03_detector_keypoint_bev.json`：各フレームの選手の足元と BEV 投影データ。
